# Data Cleaning

## Project: Customer Financial Profile Analysis

### Objective
Validate the source dataset, identify confirmed data quality issues, document any necessary changes, and save a separate processed CSV.

### Cleaning Principles
1. Preserve the original CSV.
2. Investigate issues before modifying records.
3. Avoid unnecessary deletion of customer records.
4. Record all cleaning actions.
5. Verify the final dataset before using it for EDA and SQL analysis.

### Expected Output
- data/processed/Customer_Financial_Profile_Cleaned.csv
- reports/data_cleaning_report.md

### Tools
- Python
- Pandas
- NumPy
- pathlib

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Detect the project root
cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

raw_folder = project_root / "data" / "raw"
processed_folder = project_root / "data" / "processed"
reports_folder = project_root / "reports"

# Create output directories if they don't exist
processed_folder.mkdir(parents=True, exist_ok=True)
reports_folder.mkdir(parents=True, exist_ok=True)

# Look for the standard filename or the original uploaded filename
candidates = [
    raw_folder / "Customer_Financial_Profile_Dataset.csv",
    raw_folder / "Customer_Financial_Profile_Dataset(3).csv"
]

existing_files = [p for p in candidates if p.exists()]

if len(existing_files) != 1:
    raise FileNotFoundError(
        "Expected exactly one source CSV. Check data/raw and filenames."
    )

file_path = existing_files[0]

print("Project root:", project_root)
print("Source CSV:", file_path)
print("Source exists:", file_path.exists())
print("Processed folder:", processed_folder)
print("Reports folder:", reports_folder)

Project root: e:\Customer Financial Profile Analysis
Source CSV: e:\Customer Financial Profile Analysis\data\raw\Customer_Financial_Profile_Dataset.csv
Source exists: True
Processed folder: e:\Customer Financial Profile Analysis\data\processed
Reports folder: e:\Customer Financial Profile Analysis\reports


In [2]:
# Load the original CSV
source_df = pd.read_csv(file_path)

# Make a separate working copy for cleaning
df = source_df.copy(deep=True)

print("Original rows:", len(source_df))
print("Working-copy rows:", len(df))
print("Original columns:", len(source_df.columns))
print("Working-copy columns:", len(df.columns))

display(df.head())

Original rows: 1000
Working-copy rows: 1000
Original columns: 9
Working-copy columns: 9


  Customer_ID  Age  Gender  Annual_Income  Credit_Score  Loan_Amount  \
0   CUST00000   42  Female          99848           580        13014   
1   CUST00001   44    Male          88214           634        14208   
2   CUST00002   57  Female          65358           604        12998   
3   CUST00003   67    Male         145664           784          490   
4   CUST00004   19    Male         101113           453        23056   

   Savings_Balance Employment_Status Risk_Level  
0           147042     Self-Employed     Medium  
1             9361          Employed       High  
2             7627        Unemployed     Medium  
3           110081        Unemployed       High  
4            14151     Self-Employed       High  

In [3]:
# Capture the initial data quality results

initial_missing = int(df.isna().sum().sum())
initial_duplicates = int(df.duplicated().sum())

initial_duplicate_ids = int(
    df["Customer_ID"].duplicated().sum()
)

print("Initial missing values:", initial_missing)
print("Initial exact duplicate rows:", initial_duplicates)
print("Initial duplicate customer IDs:", initial_duplicate_ids)

Initial missing values: 0
Initial exact duplicate rows: 0
Initial duplicate customer IDs: 0


In [4]:
# Check the column names and required fields

expected_columns = [
    "Customer_ID",
    "Age",
    "Gender",
    "Annual_Income",
    "Credit_Score",
    "Loan_Amount",
    "Savings_Balance",
    "Employment_Status",
    "Risk_Level"
]

print("Duplicate column names:", df.columns.duplicated().sum())

missing_columns = [
    column for column in expected_columns
    if column not in df.columns
]

unexpected_columns = [
    column for column in df.columns
    if column not in expected_columns
]

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Duplicate column names: 0
Missing expected columns: []
Unexpected columns: []


In [5]:
# Validate the data types

print(df.dtypes)

numeric_columns = [
    "Age",
    "Annual_Income",
    "Credit_Score",
    "Loan_Amount",
    "Savings_Balance"
]

text_columns = [
    "Customer_ID",
    "Gender",
    "Employment_Status",
    "Risk_Level"
]

print("\nNumeric columns:")
print(df[numeric_columns].dtypes)

print("\nText columns:")
print(df[text_columns].dtypes)

Customer_ID            str
Age                  int64
Gender                 str
Annual_Income        int64
Credit_Score         int64
Loan_Amount          int64
Savings_Balance      int64
Employment_Status      str
Risk_Level             str
dtype: object

Numeric columns:
Age                int64
Annual_Income      int64
Credit_Score       int64
Loan_Amount        int64
Savings_Balance    int64
dtype: object

Text columns:
Customer_ID          str
Gender               str
Employment_Status    str
Risk_Level           str
dtype: object


In [6]:
# Validate numeric conversions without losing information

conversion_issues = {}

for column in numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")

    # Count values that were present but could not be converted
    invalid_mask = df[column].notna() & converted.isna()

    conversion_issues[column] = int(invalid_mask.sum())

print("Invalid numeric conversions:")
print(pd.Series(conversion_issues, name="Invalid_Count"))

Invalid numeric conversions:
Age                0
Annual_Income      0
Credit_Score       0
Loan_Amount        0
Savings_Balance    0
Name: Invalid_Count, dtype: int64


In [7]:
# Check categorical values

expected_categories = {
    "Gender": {"Female", "Male"},
    "Employment_Status": {
        "Employed",
        "Self-Employed",
        "Unemployed"
    },
    "Risk_Level": {"Low", "Medium", "High"}
}

for column, allowed_values in expected_categories.items():
    observed_values = set(
        df[column].dropna().unique()
    )

    unexpected_values = observed_values - allowed_values

    print(f"\n{column}")
    print("Observed:", sorted(observed_values))
    print("Unexpected values:", sorted(unexpected_values))


Gender
Observed: ['Female', 'Male']
Unexpected values: []

Employment_Status
Observed: ['Employed', 'Self-Employed', 'Unemployed']
Unexpected values: []

Risk_Level
Observed: ['High', 'Low', 'Medium']
Unexpected values: []


In [8]:
# Validate customer identifiers

# Check duplicate identifiers
duplicate_id_count = int(
    df["Customer_ID"].duplicated().sum()
)

# Check the expected identifier format: CUST followed by five digits
id_format_ok = df["Customer_ID"].astype(str).str.fullmatch(
    r"CUST\d{5}"
)

invalid_id_count = int((~id_format_ok).sum())

print("Duplicate customer IDs:", duplicate_id_count)
print("Identifiers with unexpected format:", invalid_id_count)

if invalid_id_count > 0:
    display(df.loc[~id_format_ok, ["Customer_ID"]].head(20))

Duplicate customer IDs: 0
Identifiers with unexpected format: 0


In [9]:
# Check numerical ranges and suspicious values

# Basic sanity checks, not automatic correction rules
checks = {
    "Age <= 0": df["Age"] <= 0,
    "Age > 120": df["Age"] > 120,
    "Annual income < 0": df["Annual_Income"] < 0,
    "Loan amount < 0": df["Loan_Amount"] < 0,
    "Savings balance < 0": df["Savings_Balance"] < 0,
    "Credit score < 0": df["Credit_Score"] < 0,
}

range_check_summary = pd.Series({
    name: int(mask.sum())
    for name, mask in checks.items()
}, name="Flagged_Count")

display(range_check_summary)

Age <= 0               0
Age > 120              0
Annual income < 0      0
Loan amount < 0        0
Savings balance < 0    0
Credit score < 0       0
Name: Flagged_Count, dtype: int64

In [10]:
# Inspect possible outliers without deleting them

outlier_summary = []

for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        (df[column] < lower_bound)
        | (df[column] > upper_bound)
    )

    outlier_summary.append({
        "Column": column,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Potential_Outliers": int(mask.sum())
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
display(outlier_summary_df)

            Column  Lower_Bound  Upper_Bound  Potential_Outliers
0              Age       -8.000       96.000                   0
1    Annual_Income   -56030.625   219402.375                   0
2     Credit_Score       35.000     1117.000                   0
3      Loan_Amount   -26895.875    75089.125                   0
4  Savings_Balance  -118538.750   312419.250                   0

In [11]:
# Clean text formatting and record changes

text_columns = [
    "Customer_ID",
    "Gender",
    "Employment_Status",
    "Risk_Level"
]

whitespace_changes = {}

for column in text_columns:
    original_values = df[column].copy()

    df[column] = df[column].str.strip()

    whitespace_changes[column] = int(
        (original_values != df[column]).sum()
    )

print("Text values changed by trimming:")
print(pd.Series(whitespace_changes, name="Changed_Count"))

Text values changed by trimming:
Customer_ID          0
Gender               0
Employment_Status    0
Risk_Level           0
Name: Changed_Count, dtype: int64


In [12]:
# Remove only confirmed exact duplicate rows

rows_before_deduplication = len(df)

df = df.drop_duplicates().copy()

rows_after_deduplication = len(df)

exact_duplicates_removed = (
    rows_before_deduplication - rows_after_deduplication
)

print("Rows before:", rows_before_deduplication)
print("Rows after:", rows_after_deduplication)
print("Exact duplicates removed:", exact_duplicates_removed)

Rows before: 1000
Rows after: 1000
Exact duplicates removed: 0


In [15]:
# Apply final validation checks

# Check missing values
missing_count = int(df.isna().sum().sum())

# Check duplicate rows
duplicate_rows = int(df.duplicated().sum())

# Check unique identifiers
duplicate_ids = int(df["Customer_ID"].duplicated().sum())

# Check unexpected categories after trimming
unexpected_category_count = 0

for column, allowed_values in expected_categories.items():
    observed_values = set(df[column].dropna().unique())
    unexpected_category_count += len(
        observed_values - allowed_values
    )

# Check numeric sanity rules
range_flags = sum(
    int(mask.sum())
    for mask in checks.values()
)

final_validation = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Missing_Values": missing_count,
    "Duplicate_Rows": duplicate_rows,
    "Duplicate_Customer_IDs": duplicate_ids,
    "Unexpected_Categories": unexpected_category_count,
    "Numeric_Sanity_Flags": range_flags
}

display(pd.Series(final_validation, name="Value"))

if any([
    missing_count > 0,
    duplicate_rows > 0,
    duplicate_ids > 0,
    unexpected_category_count > 0,
    range_flags > 0
]):
    raise ValueError(
        "Validation found unresolved issues. "
        "Investigate them before saving the cleaned dataset."
    )

print("All defined final validation checks passed.")

Rows                      1000
Columns                      9
Missing_Values               0
Duplicate_Rows               0
Duplicate_Customer_IDs       0
Unexpected_Categories        0
Numeric_Sanity_Flags         0
Name: Value, dtype: int64

All defined final validation checks passed.


In [16]:
# Save the cleaned CSV

cleaned_file_path = (
    processed_folder
    / "Customer_Financial_Profile_Cleaned.csv"
)

df.to_csv(cleaned_file_path, index=False)

print("Cleaned dataset saved successfully.")
print("Output path:", cleaned_file_path)
print("Output exists:", cleaned_file_path.exists())
print("Output size:", cleaned_file_path.stat().st_size, "bytes")

Cleaned dataset saved successfully.
Output path: e:\Customer Financial Profile Analysis\data\processed\Customer_Financial_Profile_Cleaned.csv
Output exists: True
Output size: 59394 bytes


In [17]:
# Verify the saved file independently

cleaned_df = pd.read_csv(cleaned_file_path)

print("Saved rows:", len(cleaned_df))
print("Saved columns:", len(cleaned_df.columns))
print("Saved missing values:", cleaned_df.isna().sum().sum())
print("Saved duplicate rows:", cleaned_df.duplicated().sum())
print(
    "Saved duplicate customer IDs:",
    cleaned_df["Customer_ID"].duplicated().sum()
)

assert len(cleaned_df) == len(df)
assert list(cleaned_df.columns) == list(df.columns)
assert cleaned_df["Customer_ID"].is_unique

print("Saved CSV verified successfully.")

Saved rows: 1000
Saved columns: 9
Saved missing values: 0
Saved duplicate rows: 0
Saved duplicate customer IDs: 0
Saved CSV verified successfully.


## Conclusion — Data Cleaning and Validation

### Summary

In this phase, we cleaned and validated the Customer Financial Profile dataset using Python and Pandas. We preserved the original dataset and performed the cleaning operations on a separate working copy to maintain data integrity.

### Data Cleaning Activities

- **Loaded the source dataset:** Imported the original CSV file from the `data/raw/` folder.
- **Verified the dataset structure:** Checked column names, row count, column count, and data types.
- **Checked missing values:** Identified and reviewed missing values across all columns.
- **Checked duplicate records:** Examined exact duplicate rows and duplicate `Customer_ID` values.
- **Standardized text values:** Checked and removed unnecessary leading or trailing whitespace where required.
- **Validated customer IDs:** Checked whether customer identifiers followed the expected format.
- **Validated numeric values:** Reviewed financial and demographic columns for invalid or negative values.
- **Verified categorical values:** Checked the values in `Gender`, `Employment_Status`, and `Risk_Level`.
- **Saved the cleaned dataset:** Exported the validated data to `data/processed/Customer_Financial_Profile_Cleaned.csv`.
- **Reloaded and verified the output:** Confirmed that the saved CSV could be loaded successfully and retained the expected structure.

### Data Quality Results

The initial dataset inspection identified 1,000 customer records and 9 columns, with no missing values, exact duplicate rows, or duplicate customer IDs.

The cleaning process should preserve these records unless validation identifies a specific issue that justifies a documented correction or removal.

### Output File

**Cleaned dataset:** `data/processed/Customer_Financial_Profile_Cleaned.csv`

This file will be used as the input for the next phase. The original source file in `data/raw/` remains unchanged.

### Business Value

A validated dataset provides a reliable foundation for analyzing customer demographics, income, credit scores, loan amounts, savings balances, employment status, and risk categories.

### Next Steps

In the next phase, **Exploratory Data Analysis (EDA)**, we will:

1. Load the cleaned dataset.
2. Calculate key customer and financial KPIs.
3. Analyze customer age, gender, and employment distributions.
4. Explore income, credit score, loan amount, and savings distributions.
5. Compare financial indicators across risk categories.
6. Examine relationships between income, savings, and loan amounts.
7. Create visualizations and document actionable business findings.

### Final Status

**Data Cleaning and Validation: Completed**

The cleaned dataset is ready for exploratory analysis, provided all validation checks have passed and the saved output matches the notebook results.

**Note:** This project analyzes the supplied `Risk_Level` labels descriptively. It does not establish actual loan-default outcomes or prove causal relationships.